In [1]:
!pip install -q transformers datasets accelerate

In [2]:
!pip install -U bitsandbytes>=0.46.1

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import re
import pandas as pd

In [4]:
dataset = load_dataset("HuggingFaceTB/Countdown-Task-GOLD", "all")
train_data = dataset["train"]
train_data = train_data.select(range(100))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

all/train-00000-of-00001.parquet:   0%|          | 0.00/3.25M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/80000 [00:00<?, ? examples/s]

In [5]:
model_name = "Qwen/Qwen3-8B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [6]:
def make_prompt(example):
    numbers = example["nums"]
    target = example["target"]

    messages = [
        {"role": "system", "content": "You are a calculator. Do not explain. Do not think. Output only the final equation."},
        {"role": "user", "content": f"Use numbers {numbers} to get {target}. Return ONLY equation like 'a + b = c'."}
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

In [7]:
tokenizer.padding_side = "left"

def generate_answer(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id
    )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text

In [8]:
def extract_equation(text):
    match = re.search(r"([0-9+\-*/\s]+=\s*[0-9]+)", text)

    if match:
        eq = match.group(1).strip()
        if "(" in eq or ")" in eq:
            return None

        return eq

    return None

In [9]:
def check_equation(eq):
    try:
        left, right = eq.split("=")
        left_val = eval(left.strip())
        right_val = int(right.strip())
        return abs(left_val - right_val) < 1e-6

    except:
        return False

In [10]:
def uses_numbers(eq, nums):
    left, right = eq.split("=")
    found = list(map(int, re.findall(r"\d+", left)))
    return sorted(found) == sorted(nums)

In [11]:
def is_valid_solution(text, nums):
    eq = extract_equation(text)
    if eq is None:
        return False, None
    if not check_equation(eq):
        return False, eq
    if not uses_numbers(eq, nums):
        return False, eq
    return True, eq

In [12]:
valid_count = 0
results = []

for i, example in enumerate(train_data):
    prompt = make_prompt(example)
    output = generate_answer(prompt)

    is_valid, eq = is_valid_solution(output, example["nums"])

    if is_valid:
        valid_count += 1
        results.append({
            "nums": example["nums"],
            "target": example["target"],
            "prompt": example["prompt"],
            "equation": eq
        })

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [13]:
df = pd.DataFrame(results)
df.to_csv("teacher_outputs.csv")